In [9]:
import os
import chromadb
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"]="GROQ_API_KEY"
llm = ChatGroq(
    temperature=0,
    model_name="openai/gpt-oss-20b"
)
client=chromadb.Client()
collection=client.get_or_create_collection("jobs_collection")
collection.add(
    documents=[
        "Machine learning and Python AI Solutions",
        "WordPress website development services",
        "Magneto e-commerce platform development"
    ],
    metadatas=[
        {"links":"https://example.com/ml-python-portfolio"},
        {"links":"https://example.com/wordpress-portfolio"},
        {"links":"https://example.com/magneto-portfolio"},
    ],
    ids=["doc1","doc2","doc3"]
)
json_res=[
    {
        "title":"AI Engineer",
        "skills":"Python,Machine Learning,NLP,APIs",
        "description":"Hiring AI Engineer to build AI models and NLP systems.",

    }
]
job=json_res[0]
raw_links=collection.query(query_texts=job["skills"],
                           n_results=2).get("metadatas",[])
clean_links=[item["links"]for group in raw_links for item in group]
unique_links=list(set(clean_links))
prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION:
    {job_description}

    ### INSTRUCTION:
    You are Mohan, a business development executive at AtliQ.
    AtliQ is an AI & Software Consulting company dedicated to facilitating
    the seamless integration of business processes through automated tools.
    Over our experience, we have empowered numerous enterprises
    with tailored solutions, fostering scalability,
    process optimization, cost reduction, and heightened overall efficiency.
    Your job is to write a cold email to the client regarding the
    job mentioned above describing the capability of AtliQ
    in fulfilling their needs.
    Also add the most relevant ones from the following links
    to showcase Atliq's portfolio: {link_list}
    Remember you are Mohan, BDE at AtliQ.
    Do not provide a preamble.
    ### EMAIL (NO PREAMBLE):
    """
)
chain_email=prompt_email|llm
res=chain_email.invoke({"job_description":str(job),
                        "link_list":unique_links})
print(res.content)

Hi [Hiring Manager’s Name],

I’m Mohan, Business Development Executive at AtliQ. We specialize in building AI‑driven solutions that streamline operations, reduce costs, and accelerate time‑to‑market. Your posting for an AI Engineer with expertise in Python, Machine Learning, NLP, and APIs aligns perfectly with our core strengths.

**Why AtliQ?**  
- **End‑to‑end AI development**: From data ingestion and feature engineering to model deployment and monitoring, we’ve delivered production‑ready NLP and ML pipelines for Fortune 500 clients.  
- **API‑first architecture**: All our models are wrapped in scalable RESTful services, ensuring seamless integration with existing enterprise systems.  
- **Rapid prototyping & iteration**: Leveraging containerization and CI/CD, we reduce time‑to‑value by up to 60%.  
- **Domain‑agnostic expertise**: Whether it’s finance, healthcare, or e‑commerce, our solutions adapt to industry nuances while maintaining compliance and security.

**Portfolio highlight